# MATCHED COHORT - SAIGE-GENE SET-BASED
This script submits the commands for gene set-based association testing of the top genes using age-at-biospecimen collection matched cohorts with SAIGE-GENE+. The binary AML phenotype was used as the outcome variable. Multiple case-to-control matched cohorts were generated using different matching ratios, including 1:5, 1:10, 1:50, 1:100, and 1:300. Analyses included combinations of multiple minor allele frequency masks (0.01, 0.001, and 0.0001) and annotation masks (LoF, missense, and LoF+missense). The resulting summary statistics were deposited in the workspace bucket.

In [ ]:
# Load packages 
import os 
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile biosample_age_matched_analysis.sh

#!/bin/bash
set -o errexit
set -o nounset

# Loop through each ancestry  
for pop in EUR; do

    # Loop through case:control ratios 
    for ratio in 5 10 50 100 300; do
   
        echo "Processing pop: ${pop} | ratio: ${ratio} | chr: ${chrom}" 

        # --bedFile : PLINK BED genotype file containing variant dosage data
        # --bimFile : BIM file containing variant metadata and genomic positions
        # --famFile : FAM file containing sample IDs and phenotype placeholders
        # --SAIGEOutputFile : prefix for association test output files
        # --LOCO : disables leave-one-chromosome-out adjustment when set to FALSE
        # --AlleleOrder : specifies ALT allele as effect allele in association testing
        # --minMAF : minimum minor allele frequency threshold for included variants
        # --minMAC : minimum minor allele count threshold for included variants
        # --GMMATmodelFile : fitted null GLMM model from SAIGE-GENE step 1
        # --varianceRatioFile : variance ratio estimates generated during null model fitting
        # --sparseGRMFile : sparse GRM defining genetic relatedness for random effects in GLMM
        # --sparseGRMSampleIDFile : sample IDs matching GRM rows/columns to phenotype/genotype samples
        # --groupFile : gene/group annotation file defining variant sets for testing
        # --annotation_in_groupTest : functional annotation categories included in burden/SKAT tests
        # --maxMAF_in_groupTest : maximum allele frequency cutoffs used for rare variant grouping
        # --is_output_markerList_in_groupTest : outputs variant lists contributing to each tested gene/group
        # --is_fastTest : enables computationally optimized approximation for set-based tests

        step2_SPAtests.R \
          --bedFile="${plink_bed}" \
          --bimFile="${plink_bim}" \
          --famFile="${plink_fam}" \
          --SAIGEOutputFile="${output}/${pop}_matched_ratio${ratio}_high_confidence_chr${chrom}" \
          --LOCO=FALSE \
          --AlleleOrder=alt-first \
          --minMAF=0 \
          --minMAC=0.5 \
          --GMMATmodelFile="${null_glmm}/${pop}_matched_ratio${ratio}_null_glmm.rda" \
          --varianceRatioFile="${null_glmm}/${pop}_matched_ratio${ratio}_null_glmm.varianceRatio.txt" \
          --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
          --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
          --groupFile="${vat}/HC-annotation-chr${chrom}.txt" \
          --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
          --maxMAF_in_groupTest=0.0001,0.001,0.01 \
          --is_output_markerList_in_groupTest=TRUE \
          --is_fastTest=TRUE

    done
done


In [ ]:
# Copy script to workspace bucket 
!gsutil cp biosample_age_matched_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/biosample_age_matched_analysis.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out saige_gene_step02

# Submit command for chromosome 2
LOWER=2
UPPER=3
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
# Source bash script 
source ~/aou_dsub.bash 

# --image :Docker image containing SAIGE v1.5.0.2 and required dependencies
# --boot-disk-size :size (GB) of VM boot disk used for runtime environment
# --disk-size :size (GB) of attached persistent disk used for inputs and outputs
# --min-ram :minimum RAM (GB) requested for job execution
# --logging :cloud storage path where job logs (stdout/stderr) are written
# --input plink_bed :chromosome-specific PLINK BED genotype file
# --input plink_bim :chromosome-specific BIM file containing variant metadata
# --input plink_fam :chromosome-specific FAM file containing sample information
# --input-recursive grm :directory containing sparse GRM files for relatedness correction
# --input-recursive null_glmm :directory containing fitted null GLMM models and variance ratios
# --input-recursive vat :directory containing variant annotation/group definition files
# --output-recursive output :output directory for SAIGE-GENE step 2 association results
# --env chrom :environment variable passing current chromosome number into container
# --script : bash script executed inside container running SAIGE-GENE step 2

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_matched_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

# Submit command for chromosome 4
LOWER=4
UPPER=5
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
# Source bash script 
source ~/aou_dsub.bash 

# --image :Docker image containing SAIGE v1.5.0.2 and required dependencies
# --boot-disk-size :size (GB) of VM boot disk used for runtime environment
# --disk-size :size (GB) of attached persistent disk used for inputs and outputs
# --min-ram :minimum RAM (GB) requested for job execution
# --logging :cloud storage path where job logs (stdout/stderr) are written
# --input plink_bed :chromosome-specific PLINK BED genotype file
# --input plink_bim :chromosome-specific BIM file containing variant metadata
# --input plink_fam :chromosome-specific FAM file containing sample information
# --input-recursive grm :directory containing sparse GRM files for relatedness correction
# --input-recursive null_glmm :directory containing fitted null GLMM models and variance ratios
# --input-recursive vat :directory containing variant annotation/group definition files
# --output-recursive output :output directory for SAIGE-GENE step 2 association results
# --env chrom :environment variable passing current chromosome number into container
# --script : bash script executed inside container running SAIGE-GENE step 2

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_matched_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

# Submit command for chromosome 15
LOWER=15
UPPER=16
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
# Source bash script 
source ~/aou_dsub.bash 

# --image :Docker image containing SAIGE v1.5.0.2 and required dependencies
# --boot-disk-size :size (GB) of VM boot disk used for runtime environment
# --disk-size :size (GB) of attached persistent disk used for inputs and outputs
# --min-ram :minimum RAM (GB) requested for job execution
# --logging :cloud storage path where job logs (stdout/stderr) are written
# --input plink_bed :chromosome-specific PLINK BED genotype file
# --input plink_bim :chromosome-specific BIM file containing variant metadata
# --input plink_fam :chromosome-specific FAM file containing sample information
# --input-recursive grm :directory containing sparse GRM files for relatedness correction
# --input-recursive null_glmm :directory containing fitted null GLMM models and variance ratios
# --input-recursive vat :directory containing variant annotation/group definition files
# --output-recursive output :output directory for SAIGE-GENE step 2 association results
# --env chrom :environment variable passing current chromosome number into container
# --script : bash script executed inside container running SAIGE-GENE step 2

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_matched_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

# Submit command for chromosome 17 
LOWER=17
UPPER=18
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
# Source bash script 
source ~/aou_dsub.bash 

# --image :Docker image containing SAIGE v1.5.0.2 and required dependencies
# --boot-disk-size :size (GB) of VM boot disk used for runtime environment
# --disk-size :size (GB) of attached persistent disk used for inputs and outputs
# --min-ram :minimum RAM (GB) requested for job execution
# --logging :cloud storage path where job logs (stdout/stderr) are written
# --input plink_bed :chromosome-specific PLINK BED genotype file
# --input plink_bim :chromosome-specific BIM file containing variant metadata
# --input plink_fam :chromosome-specific FAM file containing sample information
# --input-recursive grm :directory containing sparse GRM files for relatedness correction
# --input-recursive null_glmm :directory containing fitted null GLMM models and variance ratios
# --input-recursive vat :directory containing variant annotation/group definition files
# --output-recursive output :output directory for SAIGE-GENE step 2 association results
# --env chrom :environment variable passing current chromosome number into container
# --script : bash script executed inside container running SAIGE-GENE step 2

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_high_confidence_analysis.sh"

done